# Silver → Gold: Build BI-Ready Fact Table
Calculate business metrics and produce the final analytics-ready fact table.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, FloatType

catalog_name = 'quickbite'

In [0]:
df = spark.table(f"{catalog_name}.silver.slv_orders")
df.limit(5).display()

### Step 1 — Calculate revenue metrics

In [0]:
# gross_amount = quantity × item_price
df = df.withColumn("gross_amount",    F.round(F.col("quantity") * F.col("item_price"), 2))

# discount_amount = gross × discount_pct / 100
df = df.withColumn("discount_amount", F.round(F.col("gross_amount") * (F.col("discount_pct") / 100), 2))

# net_amount = gross - discount + delivery_fee
df = df.withColumn("net_amount",      F.round(F.col("gross_amount") - F.col("discount_amount") + F.col("delivery_fee"), 2))

df.select("order_id","quantity","item_price","gross_amount","discount_amount","net_amount").show(5)

### Step 2 — Add derived flags and keys

In [0]:
# is_cancelled flag
df = df.withColumn("is_cancelled",
    F.when(F.col("order_status") == "Cancelled", 1).otherwise(0))

# date_id key (for joining with date dimension)
df = df.withColumn("date_id", F.date_format(F.col("dt"), "yyyyMMdd").cast(IntegerType()))

# hour_of_day (useful for peak hour analysis)
df = df.withColumn("hour_of_day", F.hour(F.col("order_ts")))

### Step 3 — Select final columns for gold table

In [0]:
gold_df = df.select(
    F.col("date_id"),
    F.col("dt").alias("order_date"),
    F.col("order_ts"),
    F.col("order_id"),
    F.col("item_seq"),
    F.col("customer_id"),
    F.col("restaurant_id"),
    F.col("agent_id"),
    F.col("item_id"),
    F.col("city"),
    F.col("payment_mode"),
    F.col("order_status"),
    F.col("is_cancelled"),
    F.col("quantity"),
    F.col("item_price"),
    F.col("discount_pct"),
    F.col("discount_amount"),
    F.col("delivery_fee"),
    F.col("gross_amount"),
    F.col("net_amount"),
    F.col("delivery_time_mins"),
    F.col("hour_of_day"),
)

gold_df.limit(5).display()

In [0]:
gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.gld_fact_orders")

print("gld_fact_orders written:", spark.table(f"{catalog_name}.gold.gld_fact_orders").count(), "rows")

### Step 4 — Create denormalised view for dashboards
This joins the fact table with all dimensions so a BI tool (Power BI / Databricks Dashboard) can query a single flat table.

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog_name}.gold.fact_orders_denorm AS (
    SELECT
        o.*,
        d.month_name,
        d.day_name,
        d.quarter,
        d.week_of_year,
        d.is_weekend,
        r.restaurant_name,
        r.cuisine_type,
        r.rating       AS restaurant_rating,
        c.name         AS customer_name,
        c.city         AS customer_city,
        a.agent_name,
        a.vehicle_type,
        m.item_name,
        m.category     AS item_category
    FROM {catalog_name}.gold.gld_fact_orders o
    JOIN {catalog_name}.gold.gld_dim_date          d ON o.date_id       = d.date_id
    JOIN {catalog_name}.gold.gld_dim_restaurants   r ON o.restaurant_id = r.restaurant_id
    JOIN {catalog_name}.gold.gld_dim_customers     c ON o.customer_id   = c.customer_id
    JOIN {catalog_name}.gold.gld_dim_delivery_agents a ON o.agent_id    = a.agent_id
    JOIN {catalog_name}.gold.gld_dim_menu_items    m ON o.item_id       = m.item_id
)
""")
print("View created: fact_orders_denorm")

### Step 5 — Sanity checks & business questions

In [0]:
# Total orders and revenue
spark.sql(f"""
    SELECT
        COUNT(*)            AS total_order_lines,
        SUM(net_amount)     AS total_revenue,
        AVG(delivery_time_mins) AS avg_delivery_mins
    FROM {catalog_name}.gold.gld_fact_orders
""").show()

In [0]:
# Revenue by city
spark.sql(f"""
    SELECT city, ROUND(SUM(net_amount),2) AS revenue, COUNT(*) AS orders
    FROM {catalog_name}.gold.gld_fact_orders
    GROUP BY city
    ORDER BY revenue DESC
""").show()

In [0]:
# Top 5 restaurants by revenue
spark.sql(f"""
    SELECT restaurant_name, cuisine_type, ROUND(SUM(net_amount),2) AS revenue
    FROM {catalog_name}.gold.fact_orders_denorm
    GROUP BY restaurant_name, cuisine_type
    ORDER BY revenue DESC
    LIMIT 5
""").show()

In [0]:
# Cancellation rate by restaurant
spark.sql(f"""
    SELECT restaurant_name,
           COUNT(*) AS total_orders,
           SUM(is_cancelled) AS cancelled,
           ROUND(SUM(is_cancelled)*100.0/COUNT(*), 1) AS cancel_pct
    FROM {catalog_name}.gold.fact_orders_denorm
    GROUP BY restaurant_name
    ORDER BY cancel_pct DESC
    LIMIT 10
""").show()

In [0]:
# Revenue by payment mode
spark.sql(f"""
    SELECT payment_mode, ROUND(SUM(net_amount),2) AS revenue, COUNT(*) AS orders
    FROM {catalog_name}.gold.gld_fact_orders
    GROUP BY payment_mode
    ORDER BY revenue DESC
""").show()